# LTSKit Cloud OCR
Chạy trên Google Colab hoặc Kaggle, với CPU hoặc GPU CUDA. Dán bốn tọa độ lấy từ LTSKit vào cell cấu hình.

Sau khi đổi runtime CPU/GPU, hãy restart runtime và chạy lại toàn bộ notebook.

In [ ]:
import os
import pathlib
import unicodedata
from difflib import SequenceMatcher

def validate_config(video_path, output_path, x0, x1, y0, y1, fps, width, height):
    video = pathlib.Path(video_path).expanduser()
    if not video.is_file(): raise ValueError('VIDEO_PATH must exist and be a file: %s' % video)
    values = (x0, x1, y0, y1)
    if any(isinstance(value, bool) or not isinstance(value, int) for value in values): raise ValueError('X0, X1, Y0, and Y1 must be finite integers')
    if isinstance(fps, bool) or not isinstance(fps, int) or fps <= 0: raise ValueError('FPS must be a positive integer')
    if not 0 <= x0: raise ValueError('ROI requires 0 <= X0')
    if not x0 < x1: raise ValueError('ROI requires X0 < X1')
    if not x1 <= width: raise ValueError('ROI requires X1 <= video_width (%d)' % width)
    if not 0 <= y0: raise ValueError('ROI requires 0 <= Y0')
    if not y0 < y1: raise ValueError('ROI requires Y0 < Y1')
    if not y1 <= height: raise ValueError('ROI requires Y1 <= video_height (%d)' % height)
    target = (pathlib.Path(output_path).expanduser() if output_path else pathlib.Path.cwd() / (video.stem + '.srt')).resolve()
    target.parent.mkdir(parents=True, exist_ok=True)
    if not os.access(target.parent, os.W_OK): raise ValueError('OUTPUT_PATH parent is not writable: %s' % target.parent)
    return str(target)

def hhmmss(seconds):
    return '%02d:%02d:%02d,%03d' % (int(seconds // 3600), int(seconds % 3600 // 60), int(seconds % 60), int(round((seconds - int(seconds)) * 1000)))

def normalize_text(text):
    return ' '.join(''.join(char if not unicodedata.category(char).startswith('P') else ' ' for char in text.lower()).split())

def text_similarity(a, b):
    a, b = normalize_text(a), normalize_text(b)
    return SequenceMatcher(None, a, b, autojunk=False).ratio() if a and b else 0.0

def merge_similar_cues(cues):
    if not cues: return []
    merged = [cues[0]]
    for start, end, text in cues[1:]:
        old_start, old_end, old_text = merged[-1]
        if 0 <= start - old_end < 1e-6 and text_similarity(old_text, text) >= 0.90:
            merged[-1] = (old_start, max(old_end, end), text if len(text) > len(old_text) else old_text)
        else: merged.append((start, end, text))
    return merged

def write_srt(cues, output_path):
    with open(output_path, 'w', encoding='utf-8') as output:
        for index, (start, end, text) in enumerate(cues, 1): output.write('%d\n%s --> %s\n%s\n\n' % (index, hhmmss(start), hhmmss(end), text))
    return len(cues)


## 1. Cài dependency
Colab: video có thể ở `/content` hoặc Google Drive đã mount. Kaggle: input thường ở `/kaggle/input/...`; output nên ở `/kaggle/working/...`.


In [ ]:
import shutil, subprocess, sys
def install_dependencies():
    if shutil.which('ffmpeg') is None:
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
    package = 'onnxruntime-gpu' if shutil.which('nvidia-smi') else 'onnxruntime'
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'], check=False, capture_output=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'rapidocr_onnxruntime', 'opencv-python-headless', package], check=True)
    print('Installed OCR dependencies with %s.' % package)
install_dependencies()


## 2. Cấu hình batch
Mỗi job có video và ROI riêng. Kaggle phải dùng `/kaggle/working/` cho `OUTPUT_DIR`; dán bốn tọa độ từ LTSKit vào từng job.


In [ ]:
PIPELINE_VERSION = 'ltskit-ocr-v2-cloud-2'
OUTPUT_DIR = '/kaggle/working/ocr-output'
FPS = 2
JOBS = [
    {'video': '/kaggle/input/my-videos/video-01.mp4', 'x0': 120, 'x1': 1800, 'y0': 760, 'y1': 1030},
]


In [ ]:
def create_ocr():
    import onnxruntime as ort
    from rapidocr_onnxruntime import RapidOCR
    providers = ort.get_available_providers()
    print('ONNX Runtime providers:', providers)
    if 'CUDAExecutionProvider' in providers:
        try:
            engine = RapidOCR(det_use_cuda=True, cls_use_cuda=True, rec_use_cuda=True)
            print('OCR device: CUDA')
            return engine, 'cuda', None
        except Exception as error:
            reason = '%s: %s' % (type(error).__name__, error)
            print('CUDA initialization failed (%s). Falling back to CPUExecutionProvider.' % reason)
            return RapidOCR(), 'cpu', reason
    print('OCR device: CPUExecutionProvider (CUDAExecutionProvider is unavailable)')
    return RapidOCR(), 'cpu', None


In [ ]:
import json, subprocess, tempfile, time
import cv2
import numpy as np
NG_DOI = 0.45
NG_TRUNG = 0.15

def white_mask(region): return cv2.inRange(cv2.cvtColor(region, cv2.COLOR_BGR2HSV), (0, 0, 200), (180, 40, 255))
def jaccard_distance(a, b):
    a, b = a > 0, b > 0
    union = int(np.count_nonzero(a | b))
    return 0.0 if union == 0 else 1.0 - int(np.count_nonzero(a & b)) / union
def probe_video(path):
    capture = cv2.VideoCapture(path)
    try:
        width, height = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if not capture.isOpened() or width <= 0 or height <= 0: raise ValueError('VIDEO_PATH cannot be opened as a video: %s' % path)
        return width, height
    finally: capture.release()
def extract_frames(path, directory, fps):
    process = subprocess.run(['ffmpeg', '-y', '-i', path, '-vf', 'fps=%d' % fps, '-q:v', '2', os.path.join(directory, 'f%06d.jpg')], capture_output=True)
    if process.returncode: raise RuntimeError('Frame extraction failed: %s' % process.stderr.decode('utf-8', 'replace')[-500:])
    frames = sorted(os.path.join(directory, name) for name in os.listdir(directory) if name.endswith('.jpg'))
    if not frames: raise RuntimeError('Frame extraction produced no images')
    return frames
def segment_frames(paths, y0, y1, x0, x1):
    segments, previous, start, motion = [], None, 0, [0.0] * len(paths)
    for index, path in enumerate(paths):
        image = cv2.imread(path)
        if image is None: continue
        mask = white_mask(image[y0:y1, x0:x1])
        if previous is not None:
            distance = jaccard_distance(mask, previous); motion[index] = distance
            if distance > NG_DOI and index > start: segments.append((start, index - 1)); start = index
        previous = mask
        if (index + 1) % 100 == 0: print('Scanning frames %d/%d' % (index + 1, len(paths)))
    return segments + [(start, len(paths) - 1)], motion
def stable_frame(motion, start, end): return min(range(start, end + 1), key=lambda index: motion[index] + (motion[index + 1] if index + 1 < len(motion) else 0))
def recognize_segments(ocr, paths, segments, motion, x0, x1, y0, y1, fps):
    cues, previous_mask, previous_text, calls, seconds = [], None, None, 0, 0.0
    for number, (start, end) in enumerate(segments, 1):
        frame_path = paths[stable_frame(motion, start, end)]; image = cv2.imread(frame_path); mask = white_mask(image[y0:y1, x0:x1])
        if previous_mask is not None and jaccard_distance(mask, previous_mask) < NG_TRUNG: text = previous_text
        else:
            began = time.perf_counter(); result, _ = ocr(frame_path); seconds += time.perf_counter() - began; calls += 1; words = []
            for box, value, score in (result or []):
                center_x = sum(point[0] for point in box) / 4; center_y = sum(point[1] for point in box) / 4
                if x0 <= center_x <= x1 and y0 <= center_y <= y1 and score > 0.5: words.append((min(point[0] for point in box), value))
            text = ' '.join(value for _, value in sorted(words)) if words else ''; previous_mask, previous_text = mask, text
        print('OCR %d%% (%d/%d): %s' % (int(number / len(segments) * 100), number, len(segments), text or '[no text]'))
        if text:
            cue_start, cue_end = start / fps, (end + 1) / fps
            if cues and cues[-1][2] == text and abs(cues[-1][1] - cue_start) < 1e-6: cues[-1] = (cues[-1][0], cue_end, text)
            else: cues.append((cue_start, cue_end, text))
    return cues, calls, seconds
def run_cloud_ocr(video_path, output_path, x0, x1, y0, y1, fps):
    total = time.perf_counter(); width, height = probe_video(video_path); output = validate_config(video_path, output_path, x0, x1, y0, y1, fps, width, height)
    began = time.perf_counter(); ocr, device, fallback = create_ocr(); model_init_ms = round((time.perf_counter() - began) * 1000)
    with tempfile.TemporaryDirectory(prefix='ltskit-ocr-') as directory:
        began = time.perf_counter(); print('Extracting frames...'); paths = extract_frames(video_path, directory, fps); extract_ms = round((time.perf_counter() - began) * 1000)
        began = time.perf_counter(); print('Scanning text changes...'); segments, motion = segment_frames(paths, y0, y1, x0, x1); segment_ms = round((time.perf_counter() - began) * 1000)
        cues, inference_calls, inference_seconds = recognize_segments(ocr, paths, segments, motion, x0, x1, y0, y1, fps)
    cue_count = write_srt(merge_similar_cues(cues), output)
    metrics = {'device': device, 'cuda_fallback_reason': fallback, 'frames': len(paths), 'segments': len(segments), 'inference_calls': inference_calls, 'cues': cue_count, 'model_init_ms': model_init_ms, 'extract_ms': extract_ms, 'segment_ms': segment_ms, 'inference_ms': round(inference_seconds * 1000), 'total_ms': round((time.perf_counter() - total) * 1000), 'output': output}
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    print('WARNING: No text was recognized. Check the ROI, subtitle color, FPS, and selected video.' if cue_count == 0 else 'SRT written to: ' + output)
    return metrics


In [ ]:
def unique_srt_path(output_dir, video_path, used):
    stem = pathlib.Path(video_path).stem
    candidate, number = stem + '.srt', 2
    while candidate in used or os.path.exists(os.path.join(output_dir, candidate)):
        candidate = '%s-%d.srt' % (stem, number); number += 1
    used.add(candidate)
    return os.path.join(output_dir, candidate)

def run_job(ocr, device, fallback_reason, job, output_path, fps):
    total = time.perf_counter()
    video_path = job['video']
    x0, x1, y0, y1 = job['x0'], job['x1'], job['y0'], job['y1']
    width, height = probe_video(video_path)
    output_path = validate_config(video_path, output_path, x0, x1, y0, y1, fps, width, height)
    with tempfile.TemporaryDirectory(prefix='ltskit-ocr-') as directory:
        began = time.perf_counter(); print('Extracting frames...'); paths = extract_frames(video_path, directory, fps); extract_ms = round((time.perf_counter() - began) * 1000)
        began = time.perf_counter(); print('Scanning text changes...'); segments, motion = segment_frames(paths, y0, y1, x0, x1); segment_ms = round((time.perf_counter() - began) * 1000)
        cues, inference_calls, inference_seconds = recognize_segments(ocr, paths, segments, motion, x0, x1, y0, y1, fps)
    cue_count = write_srt(merge_similar_cues(cues), output_path)
    metrics = {'device': device, 'cuda_fallback_reason': fallback_reason, 'frames': len(paths), 'segments': len(segments), 'inference_calls': inference_calls, 'cues': cue_count, 'model_init_ms': 0, 'extract_ms': extract_ms, 'segment_ms': segment_ms, 'inference_ms': round(inference_seconds * 1000), 'total_ms': round((time.perf_counter() - total) * 1000), 'output': output_path}
    if cue_count == 0: print('WARNING: No text was recognized. Check the ROI, subtitle color, FPS, and selected video.')
    return metrics

def write_batch_summary(output_dir, summary):
    path = os.path.join(output_dir, 'batch-summary.json')
    with open(path, 'w', encoding='utf-8') as output: json.dump(summary, output, ensure_ascii=False, indent=2)
    return path

def run_batch(jobs, output_dir, fps):
    if not jobs: raise ValueError('JOBS must contain at least one video')
    os.makedirs(output_dir, exist_ok=True)
    if not os.access(output_dir, os.W_OK): raise ValueError('OUTPUT_DIR is not writable: %s' % output_dir)
    began = time.perf_counter(); ocr, device, fallback_reason = create_ocr(); model_init_ms = round((time.perf_counter() - began) * 1000)
    summary, used = [], set()
    for index, job in enumerate(jobs, 1):
        video_path = str(job.get('video', ''))
        print('\n=== Job %d/%d: %s ===' % (index, len(jobs), video_path))
        try:
            metrics = run_job(ocr, device, fallback_reason, job, unique_srt_path(output_dir, video_path, used), fps)
            metrics['model_init_ms'] = model_init_ms if index == 1 else 0
            record = {'index': index, 'video': video_path, 'status': 'ok', **metrics}
        except Exception as error:
            record = {'index': index, 'video': video_path, 'status': 'error', 'error': '%s: %s' % (type(error).__name__, error)}
            print('Job failed:', record['error'])
        summary.append(record)
        summary_path = write_batch_summary(output_dir, summary)
    print('\nBatch summary:', summary_path)
    for record in summary: print('%d. %s - %s' % (record['index'], record['status'].upper(), record['video']))
    return summary


## 3. Chạy batch và lấy SRT
Kaggle: tất cả SRT và `batch-summary.json` sẽ nằm trong `OUTPUT_DIR` dưới `/kaggle/working`; sau khi batch xong hãy Save Version để giữ Output.


In [ ]:
BATCH_SUMMARY = run_batch(JOBS, OUTPUT_DIR, FPS)
